In [1]:
# 01_dataset_validation.ipynb
# Purpose: Validate the APL Logistics dataset before analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Setup complete")

Setup complete


In [2]:
# Load the APL Logistics dataset

file_path = "../data_raw/APL_Logistics.csv"

df = pd.read_csv(file_path, encoding="ISO-8859-1")

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully
Rows: 180519
Columns: 40


In [4]:
# Preview first 5 rows

df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,5,199.95,167.96,48.71,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,1,199.99,181.99,87.36,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


In [5]:
# View all column names

df.columns.tolist()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Fname',
 'Customer Id',
 'Customer Lname',
 'Customer Segment',
 'Customer State',
 'Customer Street',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Product Name',
 'Product Price',
 'Shipping Mode']

In [6]:
# Check data types and non-null counts

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Fname                 180519 non-null  object 
 12  Customer Id                   

In [7]:
# Check missing values

missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

missing_values

Customer Lname      8
Customer Zipcode    3
dtype: int64

In [8]:
# Check duplicate rows

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [9]:
# Unique counts for key business fields

key_columns = [
    "Customer Id",
    "Product Name",
    "Category Name",
    "Customer Segment",
    "Market",
    "Order Region",
    "Order Country",
    "Delivery Status",
    "Shipping Mode"
]

for col in key_columns:
    print(col, ":", df[col].nunique())

Customer Id : 20652
Product Name : 118
Category Name : 50
Customer Segment : 3
Market : 5
Order Region : 23
Order Country : 164
Delivery Status : 4
Shipping Mode : 4


In [10]:
# Financial validation checks

benefit_profit_diff = (df["Benefit per order"] - df["Order Profit Per Order"]).abs().max()
sales_total_diff = (df["Sales per customer"] - df["Order Item Total"]).abs().max()
sales_calc_diff = (df["Sales"] - (df["Order Item Product Price"] * df["Order Item Quantity"])).abs().max()
product_price_diff = (df["Product Price"] - df["Order Item Product Price"]).abs().max()

print("Max difference: Benefit per order vs Order Profit Per Order:", benefit_profit_diff)
print("Max difference: Sales per customer vs Order Item Total:", sales_total_diff)
print("Max difference: Sales vs Product Price × Quantity:", sales_calc_diff)
print("Max difference: Product Price vs Order Item Product Price:", product_price_diff)

Max difference: Benefit per order vs Order Profit Per Order: 0.0
Max difference: Sales per customer vs Order Item Total: 0.0
Max difference: Sales vs Product Price × Quantity: 5.684341886080802e-14
Max difference: Product Price vs Order Item Product Price: 0.0


In [11]:
# Initial KPI calculations

total_revenue = df["Sales"].sum()
total_profit = df["Order Profit Per Order"].sum()
profit_margin = (total_profit / total_revenue) * 100

total_orders = len(df)
loss_making_orders = (df["Order Profit Per Order"] < 0).sum()
loss_making_order_rate = (loss_making_orders / total_orders) * 100

total_customers = df["Customer Id"].nunique()
loss_making_customers = (
    df.groupby("Customer Id")["Order Profit Per Order"].sum() < 0
).sum()

average_discount_rate = df["Order Item Discount Rate"].mean() * 100

print("Total Revenue:", round(total_revenue, 2))
print("Total Profit:", round(total_profit, 2))
print("Profit Margin %:", round(profit_margin, 2))
print("Total Orders:", total_orders)
print("Loss-Making Orders:", loss_making_orders)
print("Loss-Making Order Rate %:", round(loss_making_order_rate, 2))
print("Total Customers:", total_customers)
print("Loss-Making Customers:", loss_making_customers)
print("Average Discount Rate %:", round(average_discount_rate, 2))

Total Revenue: 36784734.31
Total Profit: 3966902.97
Profit Margin %: 10.78
Total Orders: 180519
Loss-Making Orders: 33784
Loss-Making Order Rate %: 18.71
Total Customers: 20652
Loss-Making Customers: 4069
Average Discount Rate %: 10.17


In [17]:
# Create validation summary table

validation_summary = pd.DataFrame({
    "Validation Area": [
        "Dataset shape",
        "Missing values",
        "Duplicate rows",
        "Profit column consistency",
        "Sales total consistency",
        "Sales formula validation",
        "Date fields",
        "Shipping cost fields"
    ],
    "Finding": [
        f"{df.shape[0]} rows and {df.shape[1]} columns",
        "Only Customer Lname and Customer Zipcode have missing values",
        f"{duplicate_count} duplicate rows found",
        "Benefit per order and Order Profit Per Order match exactly",
        "Sales per customer and Order Item Total match exactly",
        "Sales equals Order Item Product Price × Order Item Quantity",
        "No date fields found",
        "No shipping cost fields found"
    ],
    "Impact on Project": [
        "Dataset is large enough for customer, product, and market-level analysis",
        "Financial analysis is not affected",
        "No duplicate-driven inflation risk",
        "Order Profit Per Order can be used as the main profit field",
        "Sales can be used as the main revenue field",
        "Revenue calculation is reliable",
        "Time trend analysis is not possible",
        "Shipping cost impact cannot be directly measured"
    ]
})

validation_summary

,Validation Area,Finding,Impact on Project
0,Dataset shape,180519 rows and 40 columns,"Dataset is large enough for customer, product,..."
1,Missing values,Only Customer Lname and Customer Zipcode have ...,Financial analysis is not affected
2,Duplicate rows,0 duplicate rows found,No duplicate-driven inflation risk
3,Profit column consistency,Benefit per order and Order Profit Per Order m...,Order Profit Per Order can be used as the main...
4,Sales total consistency,Sales per customer and Order Item Total match ...,Sales can be used as the main revenue field
5,Sales formula validation,Sales equals Order Item Product Price × Order ...,Revenue calculation is reliable
6,Date fields,No date fields found,Time trend analysis is not possible
7,Shipping cost fields,No shipping cost fields found,Shipping cost impact cannot be directly measured


## Dataset Validation Conclusion

The APL Logistics dataset contains 180,519 rows and 40 columns, with sufficient customer, product, category, market, regional, delivery, and financial fields to support profitability analysis.

The validation checks confirm that the dataset is suitable for analysis. Missing values are limited to customer last name and customer zipcode, which do not affect the main financial calculations. No duplicate rows were found, reducing the risk of inflated sales or profit figures.

Financial consistency checks show that `Benefit per order` and `Order Profit Per Order` contain identical values, so `Order Profit Per Order` will be used as the main profit field. Similarly, `Sales per customer` and `Order Item Total` match exactly, while `Sales` is consistent with product price multiplied by quantity.

Initial KPI analysis shows total revenue of 36.78M, total profit of 3.97M, and an overall profit margin of 10.78%. However, 33,784 orders are loss-making, representing 18.71% of all orders. In addition, 4,069 customers are loss-making at the aggregate customer level.

This indicates that although the business is profitable overall, profitability is uneven across orders and customers. Therefore, the next stage of analysis should investigate customer profitability, product and category profitability, discount impact, and market/regional performance.

In [12]:
# Create cleaned dataset for analysis

import os
import numpy as np

clean_df = df.copy()

# Fill minor missing values
clean_df["Customer Lname"] = clean_df["Customer Lname"].fillna("Unknown")
clean_df["Customer Zipcode"] = clean_df["Customer Zipcode"].fillna(0)

# Create simplified analysis columns
clean_df["Revenue"] = clean_df["Sales"]
clean_df["Profit"] = clean_df["Order Profit Per Order"]
clean_df["Discount_Rate"] = clean_df["Order Item Discount Rate"]
clean_df["Discount_Rate_Pct"] = clean_df["Discount_Rate"] * 100

# Order-level profit margin
clean_df["Profit_Margin_Pct"] = np.where(
    clean_df["Revenue"] != 0,
    (clean_df["Profit"] / clean_df["Revenue"]) * 100,
    0
)

# Loss-making order flag
clean_df["Is_Loss_Making_Order"] = clean_df["Profit"] < 0

# Discount bands for analysis
discount_bins = [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
discount_labels = ["0-5%", "5-10%", "10-15%", "15-20%", "20-25%", "25-30%"]

clean_df["Discount_Band"] = pd.cut(
    clean_df["Discount_Rate"],
    bins=discount_bins,
    labels=discount_labels,
    include_lowest=True
)

# Delivery risk label
clean_df["Delivery_Risk_Label"] = clean_df["Late_delivery_risk"].map({
    0: "No Late Delivery Risk",
    1: "Late Delivery Risk"
})

# Create processed data folder if it does not exist
os.makedirs("../data_processed", exist_ok=True)

# Save cleaned dataset
clean_df.to_csv("../data_processed/apl_cleaned.csv", index=False)

print("Cleaned dataset created successfully")
print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])

Cleaned dataset created successfully
Rows: 180519
Columns: 48


In [13]:
# Verify cleaned dataset was saved correctly

from pathlib import Path

cleaned_file_path = Path("../data_processed/apl_cleaned.csv")

print("Cleaned file exists:", cleaned_file_path.exists())
print("Cleaned file path:", cleaned_file_path.resolve())

Cleaned file exists: True
Cleaned file path: C:\Users\15130\Downloads\Project - API Logistics\Data_processed\apl_cleaned.csv
